In [8]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
import logging
import time
os.remove("logs/ingestion_db.log")
print("Old log deleted.")


# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s-%(levelname)s-%(message)s",
    filemode="a",
    encoding='utf-8'
)

# ── Engine (WAL mode = faster concurrent writes on SQLite) ───────────────────
engine = create_engine(
    'sqlite:///inventory.db',
    connect_args={"timeout": 30}
)

# Enable WAL mode for better write performance
with engine.connect() as conn:
    conn.execute(text("PRAGMA journal_mode=WAL;"))
    conn.execute(text("PRAGMA synchronous=NORMAL;"))

# ── Config ───────────────────────────────────────────────────────────────────
CHUNK_SIZE = 50_000      # rows per chunk  →  tune this if still slow
DATA_FOLDER = 'Data'


# ── Helper: ingest one chunk ──────────────────────────────────────────────────
def ingest_chunk(chunk, table_name, engine, first_chunk):
    """
    Write a single DataFrame chunk to SQLite.
    - First chunk: replace the table (fresh start).
    - Subsequent chunks: append without touching existing data.
    """
    exists_mode = 'replace' if first_chunk else 'append'
    chunk.to_sql(
        table_name,
        con=engine,
        if_exists=exists_mode,
        index=False,
        method='multi',      # batches INSERT statements → faster
        chunksize=1_000      # SQLAlchemy-level batch size for the INSERT
    )


# ── Main: load all CSVs in chunks ─────────────────────────────────────────────
def load_raw_data():
    start = time.time()

    for file in os.listdir(DATA_FOLDER):
        if not file.endswith('.csv'):
            continue

        file_path = os.path.join(DATA_FOLDER, file)
        table_name = file[:-4]
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

        logging.info(f"Starting ingestion: {file} ({file_size_mb:.1f} MB)")
        print(f"\n→ Ingesting: {file}  ({file_size_mb:.1f} MB)")

        chunk_number = 0
        rows_total = 0

        # ── OPTIMIZATION 1: chunksize → never loads full file into RAM ────────
        for chunk in pd.read_csv(file_path, chunksize=CHUNK_SIZE, low_memory=False):

            # ── OPTIMIZATION 2: downcast dtypes → cuts RAM per chunk in half ──
            for col in chunk.select_dtypes(include=['float64']).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast='float')
            for col in chunk.select_dtypes(include=['int64']).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast='integer')

            ingest_chunk(chunk, table_name, engine, first_chunk=(chunk_number == 0))

            rows_total += len(chunk)
            chunk_number += 1
            print(f"   chunk {chunk_number} done — {rows_total:,} rows so far", end='\r')

        logging.info(f"Done: {file} — {rows_total:,} rows in {chunk_number} chunks")
        print(f"\n   ✓ {file}: {rows_total:,} rows written to table '{table_name}'")

    total_time = (time.time() - start) / 60
    logging.info(f"Total ingestion time: {total_time:.2f} minutes")
    print(f"\nAll files done in {total_time:.2f} minutes.")


if __name__ == '__main__':
    load_raw_data()


→ Ingesting: begin_inventory.csv  (16.6 MB)
   chunk 5 done — 206,529 rows so far
   ✓ begin_inventory.csv: 206,529 rows written to table 'begin_inventory'

→ Ingesting: end_inventory.csv  (18.1 MB)
   chunk 5 done — 224,489 rows so far
   ✓ end_inventory.csv: 224,489 rows written to table 'end_inventory'

→ Ingesting: purchases.csv  (344.8 MB)
   chunk 48 done — 2,372,474 rows so far
   ✓ purchases.csv: 2,372,474 rows written to table 'purchases'

→ Ingesting: purchase_prices.csv  (1.0 MB)
   chunk 1 done — 12,261 rows so far
   ✓ purchase_prices.csv: 12,261 rows written to table 'purchase_prices'

→ Ingesting: sales.csv  (1522.8 MB)
   chunk 257 done — 12,825,363 rows so far
   ✓ sales.csv: 12,825,363 rows written to table 'sales'

→ Ingesting: vendor_invoice.csv  (0.5 MB)
   chunk 1 done — 5,543 rows so far
   ✓ vendor_invoice.csv: 5,543 rows written to table 'vendor_invoice'

All files done in 104.91 minutes.
